In [8]:
# If needed, install dependencies (run once)
# !pip install opencv-python deepface

import csv
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import cv2
import numpy as np
from deepface import DeepFace

print('Imports OK')

Imports OK


In [9]:
# ---- Config ----
DB_DIR = Path('faces_db')
CSV_PATH = Path('face_events.csv')

CAMERA_INDEX = 0
MODEL_NAME = 'Facenet512'     # other options: 'VGG-Face', ...
DB_DETECTOR = 'opencv'        # detector for DB images

THRESHOLD = 0.4             # cosine distance; lower = stricter
EXIT_AFTER_SEC = 3.0          # seconds not seen -> EXIT
MIN_FACE_PX = 80              # ignore tiny faces

ENFORCE_DETECTION_DB = False  # set True to force a face in each DB image

print('DB_DIR:', DB_DIR.resolve())
print('CSV_PATH:', CSV_PATH.resolve())

DB_DIR: C:\Users\nguye\OneDrive\Máy tính\University courses\Sem1_2026\COS30082\Week7\faces_db
CSV_PATH: C:\Users\nguye\OneDrive\Máy tính\University courses\Sem1_2026\COS30082\Week7\face_events.csv


In [10]:
SUPPORTED_IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}


@dataclass(frozen=True)
class DbEmbedding:
    name: str
    embedding: np.ndarray  # shape (d,)


def now_iso_local() -> str:
    return datetime.now().astimezone().strftime('%Y-%m-%d %H:%M:%S%z')


def ensure_csv_header(csv_path: Path) -> None:
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    if csv_path.exists() and csv_path.stat().st_size > 0:
        return
    with csv_path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['name', 'event', 'timestamp'])


def append_event(csv_path: Path, name: str, event: str, timestamp: Optional[str] = None) -> None:
    ensure_csv_header(csv_path)
    with csv_path.open('a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow([name, event, timestamp or now_iso_local()])


def l2_normalize(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    n = float(np.linalg.norm(x))
    return x / (n + eps)


def cosine_distance(a: np.ndarray, b: np.ndarray) -> float:
    a = l2_normalize(a)
    b = l2_normalize(b)
    return float(1.0 - np.dot(a, b))


def iter_face_images(db_dir: Path) -> Iterable[Tuple[str, Path]]:
    if not db_dir.exists():
        return
    for person_dir in sorted([p for p in db_dir.iterdir() if p.is_dir()]):
        name = person_dir.name
        for img_path in sorted(person_dir.rglob('*')):
            if img_path.is_file() and img_path.suffix.lower() in SUPPORTED_IMAGE_EXTS:
                yield name, img_path


def build_db_embeddings(
    db_dir: Path,
    model_name: str,
    detector_backend: str,
    enforce_detection: bool,
) -> List[DbEmbedding]:
    embeddings: List[DbEmbedding] = []
    items = list(iter_face_images(db_dir))
    if not items:
        raise FileNotFoundError(
            f'No face images found under {db_dir}. Expected structure like faces_db/Amy/*.jpg'
        )

    for name, img_path in items:
        reps = DeepFace.represent(
            img_path=str(img_path),
            model_name=model_name,
            detector_backend=detector_backend,
            enforce_detection=enforce_detection,
        )
        rep0 = reps[0] if isinstance(reps, list) else reps
        emb = np.asarray(rep0['embedding'], dtype=np.float32)
        embeddings.append(DbEmbedding(name=name, embedding=emb))

    if not embeddings:
        raise RuntimeError(f'Could not build embeddings from images under {db_dir}.')
    return embeddings


def best_match(query_embedding: np.ndarray, db: List[DbEmbedding]) -> Tuple[str, float]:
    best_name = 'Unknown'
    best_dist = float('inf')
    for item in db:
        d = cosine_distance(query_embedding, item.embedding)
        if d < best_dist:
            best_dist = d
            best_name = item.name
    return best_name, best_dist


def draw_label(
    frame: np.ndarray,
    x: int,
    y: int,
    w: int,
    h: int,
    label: str,
    color: Tuple[int, int, int],
) -> None:
    cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)
    y_text = max(0, y - 10)
    cv2.putText(
        frame,
        label,
        (x, y_text),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        color,
        2,
        cv2.LINE_AA,
    )


print('Helpers defined')

Helpers defined


In [11]:
# Build the face database embeddings
db_embeddings = build_db_embeddings(
    db_dir=DB_DIR,
    model_name=MODEL_NAME,
    detector_backend=DB_DETECTOR,
    enforce_detection=ENFORCE_DETECTION_DB,
)
print('Embeddings loaded:', len(db_embeddings))

Embeddings loaded: 6


In [14]:
# Run webcam recognition
# - Press 'q' in the video window to stop.
# - CSV will be appended with ENTER/EXIT events.

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
if face_cascade.empty():
    raise RuntimeError('Failed to load haarcascade_frontalface_default.xml')

cap = cv2.VideoCapture(CAMERA_INDEX)
if not cap.isOpened():
    raise RuntimeError(f'Could not open webcam index {CAMERA_INDEX}')

present: Dict[str, float] = {}       # name -> last_seen_time_seconds
logged_in: Dict[str, bool] = {}     # name -> ENTER logged

tick_freq = cv2.getTickFrequency()
last_cleanup = 0.0

print("Starting... press 'q' to quit.")

try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)

        t = cv2.getTickCount() / tick_freq

        for (x, y, w, h) in faces:
            if w < MIN_FACE_PX or h < MIN_FACE_PX:
                continue

            pad = int(0.10 * max(w, h))
            x0 = max(0, x - pad)
            y0 = max(0, y - pad)
            x1 = min(frame.shape[1], x + w + pad)
            y1 = min(frame.shape[0], y + h + pad)
            face_bgr = frame[y0:y1, x0:x1]

            try:
                rep = DeepFace.represent(
                    img_path=face_bgr,
                    model_name=MODEL_NAME,
                    detector_backend='skip',
                    enforce_detection=False,
                )
                rep0 = rep[0] if isinstance(rep, list) else rep
                emb = np.asarray(rep0['embedding'], dtype=np.float32)
                name, dist = best_match(emb, db_embeddings)
            except Exception:
                name, dist = 'Unknown', 999.0

            if dist <= THRESHOLD:
                label = f'{name} ({dist:.2f})'
                color = (0, 255, 0)
                present[name] = t
                if not logged_in.get(name, False):
                    append_event(CSV_PATH, name, 'ENTER')
                    logged_in[name] = True
            else:
                label = f'Unknown ({dist:.2f})'
                color = (0, 0, 255)

            draw_label(frame, x, y, w, h, label, color)

        # periodic EXIT logging
        if (t - last_cleanup) >= 0.25:
            last_cleanup = t
            for nm in list(present.keys()):
                if (t - present[nm]) >= EXIT_AFTER_SEC:
                    append_event(CSV_PATH, nm, 'EXIT')
                    present.pop(nm, None)
                    logged_in[nm] = False

        cv2.imshow('Face Recognition (q to quit)', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
finally:
    cap.release()
    cv2.destroyAllWindows()
    print('Stopped. CSV at:', CSV_PATH.resolve())

Starting... press 'q' to quit.
Stopped. CSV at: C:\Users\nguye\OneDrive\Máy tính\University courses\Sem1_2026\COS30082\Week7\face_events.csv
